# Transferarbeit Data Science — Churn-Prognose für eine gezielte Retention-Kampagne

**Modul:** Data Science · **Dozent:** Stephan Kessler  
**Student:** Lennard Bernet · **Abgabe:** `DSC_TA_Bernet_Lennard.ipynb`  
**Datensatz:** Telco Customer Churn (~7'000 Kunden, 21 Merkmale)

---

Ein Telekommunikationsanbieter verfügt über ein begrenztes Retention-Budget. Das ist die Summe an Geld oder Ressourcen, die ein Unternehmen in die Hand nimmt um bestehende Kunden vor der Kündigung zu bewahren. Als Grundlage wird ein Modell gebraucht, welches möglichst wenige echte Kündiger übersieht.
Dafür eignet sich das Telco Customer Churn Datenset, welches von IBM synthetisch generiert wurde. Es enthält keine echten Kundendaten. Diese kommen also aus dem IBM Datengenerator und waren **nie** ein Abbild eines realen Marktes.

Die Arbeit folgt den vier Blöcken der Auftragsbeschreibung. Jede Code-Zelle wird in der vorangehenden
Markdown-Zelle erläutert und jedes Resultat anschliessend interpretiert.

---
## 0 Setup und Bibliotheken

Import aller in dieser Arbeit verwendeten Bibliotheken an einer zentralen Stelle. Ein fixer `RANDOM_STATE`
sorgt dafür, dass Split, Baum-Training und Tuning reproduzierbar sind. Das ist eine Grundvoraussetzung dafür, dass
das Notebook identische Resultate liefert.

In [ ]:
# TODO: Imports ergänzen / aufräumen — hier steht bereits alles, was die Arbeit benötigt.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

---
# Block 1 — Use-Case-Analyse

*Bewertungsraster: 2 Punkte — präzise Fragestellung, Problemtyp, Limitationen.*

## 1.1 Beschreibung des Datensatzes
**Quelle:** Telco Customer Churn — https://www.kaggle.com/datasets/blastchar/telco-customer-churn  
Die Originaldatei `WA_Fn-UseC_-Telco-Customer-Churn.csv` liegt im gleichen Verzeichnis.

>Insgesamt hat der Datensatz 7043 Reihen und 21 Spalten.

Jede Reihe Repräsentiert einen Kunden. In den Spalten sind jeweils Attribute zu diesem Kunde vorhanden. Man sieht welche Services die Kunden jeweils beim Telefonanbieter haben. Ebenfalls enthalten sind Informationen wie lange der Kunde schon beim Anbieter ist und was für ein Vertrag vorliegt. Demografische Daten wie das Geschlecht, ob es alte Menschen sind, und ob sie ein Partner haben gibt es auch. Die letzte Spalte sagt noch aus, ob der Kunde im letzten Monat gekündet hat oder nicht.

## 1.2 Business-Frage

**«Welche Bestandskunden werden im kommenden Abrechnungszyklus kündigen und sollen deshalb ein
Halteangebot erhalten?»**

Mit der Spalte "Churn" hat man einen Datensatz mit dem die Kündigungen sauber dokumentiert sind. Die anderen Spalten sind Merkmale, die schon vor der Kündigung bekannt sind. Man hat jedoch keine genauen Informationen zu den Produktdaten, Netzqualität oder Preiselastizität. 
Somit hat man mit den gegebenen Daten eigentlich nur die Möglichkeit eine Vorhersage zu treffen, bei welchen Kunden im kommenden Abrechnungszyklus eine kündigung am wahrscheinlichsten ist. 
Als Massnahme bleibt allein ein Halteangebot an die gefährdeten Kunden. So ein Angebot kostet Geld und muss einzeln vergeben werden. Die eigentliche Aufgabe ist daher die Priorisierung. Ein übersehener Kündiger ist als Kunde verloren, ein überflüssiges Angebot kostet lediglich den Rabatt.

## 1.3 Problemart und Begründung

Es handelt sich um eine binäre Klassifikation im überwachten Lernen.

Die Zielvariable `Churn` hat genau zwei Ausprägungen. Nämlich `Yes` und `No` und ist in den Daten so beschriftet.
Damit ist das Kriterium für überwachtes Lernen erfüllt. Das Modell lernt aus Beispielen, zu denen die richtige Antwort bekannt ist.
Eine Regression kommt nicht in Frage, da keine stetikge Zielgrösse vorhergesagt werden soll.
Ein Clustering-Verfahren wie K-Means eignet sich ebenfalls nicht. Das arbeitet mit Labels und Gruppen die selbstständig entdeckt werden. Hier kennt man diese Gruppe aber schon anhand von `Churn`.
Es wird auch nicht nur eine Klassenzuordnung sondern auch eine Wahrscheinlichkeit benötigt. Nur damit lässt sich schlussendlich eine gute Rangliste für die Priorität bilden um das begrenzte Budget auf die gefährdetsten Kunden zu lenken.

## 1.4 Nutzen für das Unternehmen

Ein Halteangebot kostet einer Firma bedeutend weniger wie eine Neuakquisition eines Kunden. Akquisition heisst Werbung, Vertrieb, Startrabatte etc.
Ein Rabatt für ein aktueller Kunde ist ein einmaliger, kalkulierbarer Abzug. Ein verlorener Kunde ist ein entgangener Umsatz den man so einfach nicht zurück bekommt.
Das Budget in der Firma ist begrenzt, also braucht es eine Rangliste anstatt einfach blindes Verteilen von Halteangeboten. Ganau das liefert das Modell.
Ein übersehener Kündiger ist weg und nicht mehr korrigierbar, ein überflüssiges Angebot kostet nur den Rabatt.

Annahme Monatsumsatz eines Kunden:        CHF  60
Annahme Restlaufzeit nach Rettung:         12 Monate
→ Wert eines gehaltenen Kunden:           CHF 720

Annahme Halteangebot: 20 % Rabatt, 6 Monate
→ Kosten des Angebots:                    CHF  72
Verhältnis Nutzen zu Kosten:                1 : 10

Diese Zahlen sind nur mal grobe Annahmen, die tatsächlichen Umsätze werden in Block 2 aus den Daten ermittelt.

## 1.5 Risiken sowie ethische und technische Grenzen

1. **Diskriminierung:** Die Spalten `gender` und `SeniorCitizen` sind eher geschütze/sensible Merkmale. Wenn man Rabatte danach verteilt bekommen die Personen unterschiedliche Preise, nur wegen eines Merkmals, das sie nicht ändern können. Es ist rechtlich gesehen recht heikel und ethisch nicht komplett vertretbar. Wenn man diese Spalten komplett ausschliesst ist das Problem nur abgeschwächt. Merkmale wie `partner` oder `paymentMethod` können mit Alter und Lebessituationen zusammenhängen, die das entfernte Merkmal teilweise ersetzen.

2. **Korrelation ist nicht Kausalität** Das Modell lernt Muster und nicht Wirkmechanismen. Es sagt aus wer kündigt aber nicht warum. Wenn man also sagt "Merkmal X sagt Churn voraus, also ändern wir X" ist gefährlich. Wenn man Kunden zum Beispiel zwangsweise in Jahresverträge drängt, weil Monatsverträge einen hohen Churn Wert haben ändert das nicht unbedingt die Wechselbereitschaft zu einem anderen Anbieter. Ursachen wie Netzqualität, Störungen, Beschwerden und Konkurrenzangebote fehlen in den Daten. Das Modell taugt also wirklich nur zur Priorisierung und nicht als Begründung für Produktentscheide.

3. **Data Leakage** Merkmale, die erst nach der durch die Kündigung entstehen, dürfen nicht ins Modell. Sicher die customerID gehört raus, da sie keinen fachlichen Gehalt hat.
Beim Ablauf des Trainings kann auch Data Leakage entstehen. Man darf Informationen aus dem Testset nicht in das Training fliessen lassen. Deshalb wird in dieser Arbeit zuerst aufgeteilt und die gesamte Vorverarbeitung in eine Pipeline gepackt, die ausschliesslich auf den Trainingsdaten gefittet wird. Ein Data Leakage ist auch ein Wirtschaftliches Risiko, da die Resultate eine fehlerhafte Priorisierung generieren würden.

4. **Momentaufnahme / Data Drift** Der Datensatz bildet einen einzigen Stichtag ab und enthält keine Zeitachse. Verändert sich der Markt verschieben sich die Merkmalsverteilungen als auch der Zusammenhang zwischen Merkmalen und Kündigungen. Ein einmal trainiertes Modell verliert dadurch mit der Zeit an Wert. Im Betrieb bräuchte es deshalb eine laufende Überwachung und ein ständiges Neutrainieren. Die Daten sind sowieso synthetisch von IBM erzeugt worden. Die Method dieser Arbeit ist übertragbar, die konkreten Werte sind es aber nicht.

---
# Block 2 — Datenprozessierung und explorative Datenanalyse

*Bewertungsraster: 8 Punkte — Import/Überblick, begründete Bereinigung, EDA und Feature Engineering.
Der punktestärkste Block der Arbeit.*

## 2.1 Import und erster Überblick

Der Datensatz wird über einen **relativen Pfad** geladen, damit das Notebook ohne Internetverbindung
vollständig durchläuft. Anschliessend verschaffen `shape`, `head()`, `info()` und `describe()` den vom
Auftrag geforderten Überblick über Umfang, Datentypen und Wertebereiche.

In [ ]:
# TODO: Pfad prüfen — CSV muss im selben Ordner wie dieses Notebook liegen.
DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(DATA_PATH)

print("Form des Datensatzes (Zeilen, Spalten):", df.shape)
df.head()

In [ ]:
# TODO: Ausgabe im Markdown darunter interpretieren.
df.info()

In [ ]:
# TODO: describe() für numerische UND kategoriale Spalten ausgeben.
display(df.describe())
display(df.describe(include="object"))

> **TODO — Interpretation (der wichtigste Teil dieses Abschnitts):** Auf zwei Auffälligkeiten hinweisen,
> die sich direkt aus `info()` ergeben:
> 1. `TotalCharges` ist als `object` eingelesen, obwohl es sich um einen Geldbetrag handelt — es steckt
>    also ein nicht-numerischer Wert in der Spalte.
> 2. `SeniorCitizen` ist bereits als 0/1 codiert und damit faktisch kategorial, obwohl es numerisch wirkt.
>
> Beides wird in 2.2 behandelt.

## 2.2 Qualitätsprüfung und begründete Bereinigung

Geprüft werden die drei im Auftrag genannten Aspekte: fehlende Werte, Dubletten und Ausreisser.
Zu jedem Befund gehört ein **begründeter** Entscheid — Bereinigen ohne Begründung gibt Abzug.

In [ ]:
# --- Fehlende Werte ---
print("Fehlende Werte je Spalte:")
print(df.isna().sum()[lambda s: s > 0])

# TODO: TotalCharges gezielt untersuchen — die Spalte enthält Leerstrings statt echter NaN.
# Vorgehen: pd.to_numeric(..., errors="coerce"), danach zählen, welche Zeilen betroffen sind,
# und prüfen, welchen Wert 'tenure' bei genau diesen Zeilen hat.

> **TODO — Begründeter Entscheid:** Die betroffenen Zeilen sind Kunden mit `tenure == 0`, also
> Neukunden, die noch keine Rechnung erhalten haben. Der fehlende Wert ist damit **kein Messfehler,
> sondern inhaltlich erklärbar**. Zwei Optionen gegeneinander abwägen und eine wählen:
> Imputation mit `0` (fachlich korrekt: es wurde noch nichts bezahlt) oder Entfernen der Zeilen
> (vertretbar, da es sich um einen sehr kleinen Anteil handelt und Neukunden ohne Historie für die
> Fragestellung ohnehin wenig aussagen). **Entscheid und Begründung ausformulieren.**

In [ ]:
# --- Dubletten ---
# TODO: Zwei Ebenen prüfen und beide im Markdown kommentieren:
#   a) doppelte customerID  -> df["customerID"].duplicated().sum()
#   b) vollständig identische Zeilen -> df.duplicated().sum()

In [ ]:
# --- Ausreisser ---
# TODO: Boxplots oder IQR-Regel (Folie 02: Quartile, Interquartilsabstand) auf die drei
# numerischen Spalten tenure, MonthlyCharges, TotalCharges anwenden.

> **TODO — Begründeter Entscheid:** Bei diesem Datensatz zeigen die numerischen Spalten keine unplausiblen
> Extremwerte; hohe `TotalCharges` gehören zu langjährigen Kunden und sind fachlich korrekt. Der Entscheid
> lautet deshalb bewusst **«keine Entfernung»** — mit der Begründung, dass ein statistischer Ausreisser nach
> IQR-Regel nicht automatisch ein Fehler ist. Genau diese Unterscheidung wird in der Bewertung gesucht.
>
> Zusätzlich: `customerID` ist ein reiner Identifikator ohne Vorhersagekraft und wird vor dem Modelltraining
> entfernt — andernfalls würde das Modell auswendig lernen statt zu verallgemeinern.

In [ ]:
# TODO: Bereinigungsschritte hier gebündelt und nachvollziehbar ausführen.
# Empfehlung: auf einer Kopie arbeiten (df_clean = df.copy()), damit der Rohzustand oben erhalten bleibt.

## 2.3 Explorative Datenanalyse

Gefordert sind mindestens drei beschriftete und interpretierte Visualisierungen. Die drei gewählten
Grafiken sind nicht beliebig, sondern beantworten je eine Frage, die für den roten Faden nötig ist:
Wie stark ist die Zielvariable unausgeglichen? Wann kündigen Kunden? Und welches Merkmal trennt am besten?

### Visualisierung 1 — Verteilung der Zielvariable

Diese Grafik liefert die **Baseline**: Sagt ein triviales Modell für alle Kunden «keine Kündigung» voraus,
erreicht es bereits eine hohe Accuracy. Damit ist die Accuracy als Leitmetrik entwertet — die Begründung
für die Metrikwahl in Block 4 beginnt hier.

In [ ]:
# TODO: Balkendiagramm der Churn-Verteilung (absolut und in Prozent).
# Pflicht: Titel, Achsenbeschriftungen. Prozentwert der Minderheitsklasse im Text festhalten.

> **TODO — Interpretation:** Anteil der Kündiger nennen und daraus die Konsequenz ableiten
> (Baseline-Accuracy = Anteil der Mehrheitsklasse; `stratify` beim Split; Recall als Leitmetrik).

### Visualisierung 2 — Kundendauer (`tenure`) nach Churn-Status

Zeigt, **wann** im Kundenlebenszyklus gekündigt wird, und begründet damit das erste neue Feature in 2.4.

In [ ]:
# TODO: Histogramm von tenure, getrennt nach Churn (hue="Churn").
# Pflicht: Titel, Achsenbeschriftungen, Legende.

> **TODO — Interpretation:** Auf die Häufung der Kündigungen in den ersten Monaten hinweisen und daraus
> die Handlungsempfehlung ableiten, dass Retention-Massnahmen früh ansetzen müssen.

### Visualisierung 3 — Kündigungsrate nach Vertragsart (`Contract`)

Zeigt das trennschärfste kategoriale Merkmal und liefert die fachliche Erklärung für die spätere
Modell-Interpretation.

In [ ]:
# TODO: Balkendiagramm der Churn-Rate je Contract-Ausprägung (Month-to-month / One year / Two year).
# Pflicht: Titel, Achsenbeschriftungen, Rate in Prozent.

> **TODO — Interpretation:** Unterschied zwischen Monatsvertrag und Jahresverträgen benennen und
> ausdrücklich festhalten, dass daraus **keine Kausalität** folgt (Rückgriff auf Punkt 1.5).

## 2.4 Feature Engineering

Gefordert sind mindestens zwei neue, begründete Merkmale. Die beiden gewählten stützen sich direkt auf
die im Unterricht behandelten Muster «Verhältnisse» und «Aggregationen».

**Feature 1 — `avg_charge_per_month` (Verhältnis)**

`TotalCharges / tenure` ergibt den tatsächlich bezahlten Durchschnittsbetrag pro Monat. Weicht dieser Wert
vom aktuellen `MonthlyCharges` ab, hat sich der Tarif des Kunden über die Zeit verändert — eine
Preiserhöhung ist ein plausibler Kündigungsauslöser, der in keiner Einzelspalte sichtbar ist.

**Feature 2 — `num_services` (Aggregation)**

Anzahl der gebuchten Zusatzdienste (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`,
`StreamingTV`, `StreamingMovies`). Die Hypothese: Je stärker ein Kunde in das Ökosystem eingebunden ist,
desto höher die Wechselhürde. Die Aggregation verdichtet sechs Spalten in eine interpretierbare Kennzahl.

In [ ]:
# TODO: Beide Features berechnen.
# Achtung bei Feature 1: Division durch tenure == 0 abfangen (je nach Entscheid in 2.2 bereits erledigt).
# Achtung bei Feature 2: Ausprägungen 'No internet service' zählen NICHT als gebuchter Dienst.

In [ ]:
# TODO: Plausibilisierung der neuen Features — describe() und ein kurzer Vergleich
# der Mittelwerte je Churn-Gruppe (df.groupby("Churn")[[...]].mean()).

> **TODO — Interpretation:** Kurz belegen, dass die neuen Merkmale tatsächlich zwischen den Gruppen
> unterscheiden, und damit rechtfertigen, dass sie ins Modell aufgenommen werden.

## 2.5 Train/Test-Split

Der Split erfolgt **vor** jeder Skalierung und Codierung. Würden Mittelwert und Standardabweichung auf dem
gesamten Datensatz berechnet, flössen Verteilungsinformationen der Testdaten ins Training ein — genau das
in der Vorlesung behandelte **Data Leakage**. Alle Transformationen werden deshalb ab Block 3 in einer
`Pipeline` gekapselt, die ausschliesslich auf den Trainingsdaten gefittet wird.

`stratify=y` erhält das Klassenverhältnis aus Visualisierung 1 in beiden Teilmengen — bei einer
unausgeglichenen Zielvariable ist das zwingend, sonst schwanken die Testresultate zufällig.

In [ ]:
# TODO: Zielvariable in 0/1 umwandeln, Feature-Matrix X und Zielvektor y bilden,
# customerID (und gemäss Entscheid aus 1.5 ggf. gender) entfernen.

# X = ...
# y = ...

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# TODO: Form der Teilmengen und Klassenverhältnis in beiden Sets ausgeben und kurz kommentieren.

---
# Block 3 — Modellwahl und Training

*Bewertungsraster: 7 Punkte — Begründung, korrektes Training, dokumentiertes Tuning.*

## 3.1 Auswahl und Begründung der beiden Modelle

Bewusst gewählt wurden zwei Modelle mit **grundsätzlich verschiedenem Ansatz**, damit der Vergleich in
Block 4 eine Aussage erlaubt:

| | Logistische Regression | Entscheidungsbaum |
|---|---|---|
| Ansatz | linear, parametrisch | regelbasiert, nicht-linear |
| Ergebnis | Wahrscheinlichkeit | Klassenzuordnung über Blätter |
| Stärke | interpretierbare Koeffizienten, gute Wahrscheinlichkeiten für eine Priorisierung | erfasst Interaktionen und Schwellenwerte ohne Vorgabe |
| Schwäche | modelliert nur lineare Zusammenhänge in den Log-Odds | neigt ohne Begrenzung stark zu Overfitting |
| Skalierung nötig | ja | nein |

> **TODO (2–3 Sätze):** Ergänzen, weshalb gerade dieses Paar zur Fragestellung passt: Die logistische
> Regression liefert die für die Budgetpriorisierung benötigte Wahrscheinlichkeit, der Baum dient als
> nicht-linearer Gegenspieler und prüft, ob im Datensatz Zusammenhänge stecken, die ein lineares Modell
> nicht erfassen kann.

## 3.2 Funktionsweise in Kürze

> **TODO — je 3–4 Sätze, in eigenen Worten (dieser Abschnitt kommt in der Befragung mit hoher
> Wahrscheinlichkeit dran):**
>
> **Logistische Regression:** gewichtete Summe der Merkmale → Sigmoid-Funktion → Wert zwischen 0 und 1,
> interpretierbar als Kündigungswahrscheinlichkeit. Der Schwellenwert (standardmässig 0.5) übersetzt diese
> Wahrscheinlichkeit in eine Klasse. Die Gewichte werden so bestimmt, dass die beobachteten Labels möglichst
> wahrscheinlich werden.
>
> **Entscheidungsbaum:** teilt die Daten rekursiv anhand jeweils eines Merkmals so auf, dass die
> entstehenden Gruppen möglichst rein bezüglich der Zielvariable sind (Gini-Kriterium). Ohne
> Tiefenbegrenzung wächst der Baum, bis jedes Blatt rein ist — und lernt damit das Rauschen der
> Trainingsdaten auswendig.

## 3.3 Vorverarbeitung als Pipeline

Numerische Merkmale werden z-standardisiert (Folie 04), kategoriale per One-Hot codiert — bewusst kein
Label-Encoding, da dieses eine nicht existierende Rangordnung suggerieren würde. Beides steckt in einem
`ColumnTransformer` innerhalb der Pipeline, damit der Fit garantiert nur auf den Trainingsdaten geschieht.

In [ ]:
# TODO: Spaltenlisten bilden und ColumnTransformer aufsetzen.
# num_features = [...]   # tenure, MonthlyCharges, TotalCharges, avg_charge_per_month, num_services
# cat_features = [...]   # alle übrigen object-Spalten

# preprocessor = ColumnTransformer([
#     ("num", StandardScaler(), num_features),
#     ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_features),
# ])

## 3.4 Modell A — Logistische Regression

In [ ]:
# TODO: Pipeline aus preprocessor + LogisticRegression(max_iter=1000, random_state=RANDOM_STATE) bauen
# und auf den Trainingsdaten fitten.

## 3.5 Modell B — Entscheidungsbaum

In [ ]:
# TODO: Pipeline aus preprocessor + DecisionTreeClassifier(random_state=RANDOM_STATE) bauen und fitten.
# Zusatz: einmal ohne Tiefenbegrenzung trainieren und Trainings- gegen Testscore halten —
# der Abstand ist der Beleg für Overfitting und die Motivation für das Tuning im nächsten Schritt.

## 3.6 Hyperparameter-Tuning

Das Tuning erfolgt mit `GridSearchCV` und 5-facher Kreuzvalidierung auf den **Trainingsdaten**; das Testset
bleibt bis Block 4 unberührt.

Entscheidend und explizit zu begründen: Als Optimierungsmetrik wird **nicht** die Accuracy verwendet,
sondern `recall` (bzw. `f1`). Damit wird das Tuning auf dasselbe Ziel ausgerichtet wie die Business-Frage —
möglichst wenige Kündiger übersehen. Wer hier auf Accuracy optimiert, optimiert am Geschäftsproblem vorbei.

In [ ]:
# TODO: Suchraum definieren und GridSearchCV ausführen.
# Baum:   max_depth [3, 5, 7, 10, None], min_samples_leaf [1, 10, 50]
# LogReg: C [0.01, 0.1, 1, 10], class_weight [None, "balanced"]
# GridSearchCV(..., cv=5, scoring="recall", n_jobs=-1)

# TODO: beste Parameter und besten CV-Score ausgeben.

> **TODO — Interpretation:** Beste gefundene Parameter nennen und erklären, *warum* sie plausibel sind
> (z. B. begrenzte Baumtiefe = weniger Overfitting; `class_weight="balanced"` gewichtet die seltene
> Kündiger-Klasse stärker und hebt damit den Recall).

## 3.7 Vorhersagen auf dem Testset

In [ ]:
# TODO: Für beide (getunten) Modelle predict() und predict_proba() auf X_test berechnen
# und für Block 4 bereitstellen.

---
# Block 4 — Evaluation und Fazit

*Bewertungsraster: 7 Punkte — passende Metriken, Modellvergleich, Interpretation, mindestens drei
Verbesserungsmöglichkeiten.*

## 4.1 Wahl der Metriken

Für eine Klassifikation mit unausgeglichener Zielvariable ist die Accuracy allein irreführend: Das triviale
Modell «niemand kündigt» erreicht bereits den in Visualisierung 1 ermittelten Mehrheitsanteil, ohne einen
einzigen Kündiger zu finden. Berichtet werden deshalb:

- **Recall** — Anteil der tatsächlichen Kündiger, die erkannt wurden. **Leitmetrik dieser Arbeit**, weil ein
  übersehener Kündiger den vollen Kundenwert kostet.
- **Precision** — Anteil der Vorhersagen «Kündigung», die zutreffen. Steht für die Effizienz des
  Retention-Budgets.
- **F1-Score** — harmonisches Mittel beider; nötig, weil Recall allein durch «alle als Kündiger einstufen»
  trivial maximierbar wäre.
- **ROC-AUC** — schwellenwertunabhängige Trennschärfe, sinnvoll für den Modellvergleich.
- **Accuracy** — nur zur Einordnung gegenüber der Baseline.

In [ ]:
# TODO: Hilfsfunktion, die für ein Modell alle Metriken berechnet und als dict/Series zurückgibt.

## 4.2 Modellvergleich

In [ ]:
# TODO: Metriken beider Modelle in einem DataFrame gegenüberstellen (Zeilen = Modelle, Spalten = Metriken)
# und zusätzlich die Baseline-Accuracy als Referenzzeile aufnehmen.

In [ ]:
# TODO: Konfusionsmatrizen beider Modelle nebeneinander plotten (ConfusionMatrixDisplay, plt.subplots).
# Pflicht: Titel und beschriftete Achsen mit den Klassenlabels.

## 4.3 Interpretation der Ergebnisse

> **TODO — die vier Felder der Konfusionsmatrix fachlich übersetzen, nicht nur benennen:**
>
> - **False Negative:** Kunde kündigt, wurde aber nicht erkannt → kein Halteangebot → Kunde verloren.
>   Teuerster Fehler.
> - **False Positive:** Kunde wäre geblieben, erhält aber ein Angebot → Rabattkosten ohne Not.
>   Ärgerlich, aber vergleichsweise günstig.
> - Daraus folgt der Entscheid, welches Modell gewinnt — und zwar **nicht** zwingend jenes mit der höchsten
>   Accuracy.
>
> Zusätzlich den **Schwellenwert als Stellhebel** diskutieren: Über `predict_proba` lässt sich die Schwelle
> von 0.5 senken (z. B. auf 0.35), was den Recall zulasten der Precision erhöht. Die richtige Schwelle ist
> eine betriebswirtschaftliche Entscheidung und keine statistische.

In [ ]:
# TODO (optional, aber überzeugend und mit wenig Aufwand): Recall und Precision des besseren Modells
# für mehrere Schwellenwerte berechnen und in einer kleinen Tabelle oder einem Plot zeigen.

## 4.4 Beantwortung der Business-Frage

> **TODO (4–6 Sätze):** Die in 1.2 gestellte Frage explizit beantworten. Enthalten sein müssen:
> welches Modell eingesetzt wird, welcher Anteil der Kündiger damit erkannt wird, wie viele Angebote dafür
> insgesamt verschickt werden müssen — und ein klarer Satz dazu, ob das Ergebnis für einen produktiven
> Einsatz ausreicht oder nur als Entscheidungsunterstützung taugt.

## 4.5 Verbesserungsmöglichkeiten

> **TODO — mindestens drei ausformulieren (je 2–3 Sätze, konkret statt allgemein):**
>
> 1. **Umgang mit der Klassen-Unwucht vertiefen:** Über `class_weight="balanced"` hinaus Resampling-Verfahren
>    wie SMOTE prüfen und deren Effekt auf Recall und Precision messen.
> 2. **Kostenbasierte Optimierung statt symmetrischer Metrik:** Eine Kostenmatrix mit den effektiven Werten
>    für Kundenverlust und Rabatt hinterlegen und den Schwellenwert auf den erwarteten Deckungsbeitrag
>    statt auf F1 optimieren.
> 3. **Zusätzliche Datenquellen erschliessen:** Supportkontakte, Zahlungsverzug, Störungsmeldungen und
>    Nutzungsintensität fehlen im Datensatz vollständig, sind aber erfahrungsgemäss starke Frühindikatoren.
> 4. **Stärkere Modellklassen prüfen:** Random Forest oder Gradient Boosting als Ergänzung, mit
>    Kreuzvalidierung sauber gegen die bestehenden Modelle gestellt.
> 5. **Betrieb mitdenken:** Periodisches Neutraining und Monitoring der Vorhersagequalität, um Data Drift
>    zu erkennen.

---
# 5. KI-Klassifizierung und Hilfsmittelverzeichnis

Gemäss Auftrag sind eingesetzte KI-Werkzeuge transparent auszuweisen.

| Hilfsmittel | Version / Modell | Verwendungszweck | Umfang |
|---|---|---|---|
| *TODO* | | | |

> **TODO:** Ehrlich und konkret ausfüllen — wofür genau ein KI-Werkzeug eingesetzt wurde (z. B. Strukturierung
> des Notebooks, Code-Vorschläge, sprachliche Überarbeitung) und was eigenständig erarbeitet wurde.
> Ebenso aufführen: verwendete Bibliotheken sowie die Unterrichtsunterlagen.

**Quellen**

- Telco Customer Churn Dataset, IBM Sample Data via Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Unterrichtsunterlagen Data Science, TEKO, S. Kessler: Einführung, Datenverarbeitung, Mathematische Werkzeuge, Modelle/Methoden/Algorithmen
- scikit-learn Dokumentation: https://scikit-learn.org